In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/UNSW_NB15_training-set.csv")
test = pd.read_csv("../data/UNSW_NB15_testing-set.csv")
print(train.shape, test.shape)

(175341, 45) (82332, 45)


In [2]:
for df in (train, test):
    df.drop(columns=["id"], inplace=True, errors="ignore")

train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)
print(train.shape, test.shape)

C:\Users\Mega PC\AppData\Local\Temp\ipykernel_10068\3304596930.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["id"], inplace=True, errors="ignore")
C:\Users\Mega PC\AppData\Local\Temp\ipykernel_10068\3304596930.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["id"], inplace=True, errors="ignore")


(107740, 44) (55945, 44)


In [3]:
# Réponses (à ne JAMAIS donner au modèle comme entrée)
y_train = train["label"]
y_test = test["label"]
y_train_cat = train["attack_cat"]
y_test_cat = test["attack_cat"]

# Entrées
X_train = train.drop(columns=["label", "attack_cat"])
X_test = test.drop(columns=["label", "attack_cat"])
print(X_train.shape, X_test.shape)

(107740, 42) (55945, 42)


In [5]:
from sklearn.preprocessing import OrdinalEncoder

cat_cols = ["proto", "service", "state"]

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

In [6]:
print(X_train.dtypes.value_counts())
print(X_train.isnull().sum().sum(), "valeurs manquantes")
print(np.isinf(X_train).sum().sum(), "valeurs infinies")

int64      28
float64    14
Name: count, dtype: int64
0 valeurs manquantes
0 valeurs infinies


In [7]:
import os
os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)
y_train_cat.to_csv("../data/processed/y_train_cat.csv", index=False)
y_test_cat.to_csv("../data/processed/y_test_cat.csv", index=False)

## Résumé du prétraitement

**Jeu de données :** UNSW-NB15 (fichiers training-set et testing-set déjà séparés).

**Nettoyage**
- Colonne `id` supprimée : simple numéro de ligne, sans information utile.
- Doublons supprimés : X dans le train, X dans le test.
- Aucune valeur manquante ni infinie détectée.

**Séparation entrées / réponses**
- Réponses : `label` (0 = normal, 1 = attaque) et `attack_cat` (type d'attaque).
- Ces deux colonnes sont retirées de `X` pour éviter une fuite de données
  (le modèle ne doit pas voir la réponse).

**Encodage**
- Colonnes textuelles `proto`, `service` et `state` transformées avec `OrdinalEncoder`.
- Ajusté uniquement sur le train, puis appliqué au test ;
  les catégories inconnues reçoivent la valeur -1.
- One-Hot écarté : `proto` contient trop de valeurs distinctes, ce qui créerait
  un très grand nombre de colonnes.

**Normalisation**
- Non appliquée : Random Forest et XGBoost sont des modèles à base d'arbres,
  insensibles à l'échelle des variables.

**Résultat :** X_train de forme (X, X), X_test de forme (X, X), sauvegardés dans `data/processed/`.

**Prochaine étape :** entraînement de Random Forest et XGBoost, puis évaluation
(Precision, Recall, F1, matrice de confusion).